# Pandas Merge Pitfalls: Silent Row Explosion & Lost Data

Merges are the most common source of wrong row counts in pandas. A single duplicate key
can silently multiply your rows, a mismatched dtype can produce zero matches, and a wrong
join type can drop data without any error. This notebook walks through the three most
dangerous pitfalls and shows defensive patterns to catch them early.

In [ ]:
!pip install pandas numpy --quiet

## Generate Synthetic Data

We create two tables that look perfectly normal at first glance:
- **orders**: 1 000 orders with `customer_id`, `order_id`, and `amount`.
- **customers**: 200 customer records with `customer_id`, `name`, and `segment`.

Hidden traps baked in:
1. Some customers appear **twice** in the `customers` table (duplicate keys).
2. Some orders reference a `customer_id` that does **not** exist in `customers`.
3. A second version of `customers` stores `customer_id` as a **string** instead of int.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Orders table (1 000 rows) ---
orders = pd.DataFrame({
    "order_id": range(1, 1001),
    "customer_id": np.random.randint(1, 220, size=1000),   # ids 201-219 won't exist in customers
    "amount": np.round(np.random.uniform(5, 500, size=1000), 2),
})

# --- Customers table (200 unique + 10 duplicates = 210 rows) ---
unique_customers = pd.DataFrame({
    "customer_id": range(1, 201),
    "name": [f"Customer_{i}" for i in range(1, 201)],
    "segment": np.random.choice(["Bronze", "Silver", "Gold"], size=200),
})

# Duplicate 10 customers (the trap)
duplicates = unique_customers.sample(10, random_state=42).copy()
duplicates["segment"] = "Platinum"  # same id, different segment
customers = pd.concat([unique_customers, duplicates], ignore_index=True)

print(f"orders:    {len(orders):>6} rows")
print(f"customers: {len(customers):>6} rows  (200 unique ids + 10 duplicates)")
display(customers[customers.duplicated(subset="customer_id", keep=False)]
        .sort_values("customer_id").head(10))

## Pitfall 1: Duplicate Keys Cause Silent Row Explosion

When the right table has duplicate keys, a merge creates a **cartesian product** for
every matching key. The result has more rows than either input — and pandas raises
**no error** by default.

In [ ]:
# Naive merge — looks harmless
merged = orders.merge(customers, on="customer_id", how="left")

print(f"orders before merge: {len(orders)} rows")
print(f"merged result:       {len(merged)} rows  <-- MORE rows than we started with!")
print(f"Extra rows created:  {len(merged) - len(orders)}")

# Show the explosion for one duplicated customer
dup_id = duplicates["customer_id"].iloc[0]
print(f"\nExample: customer_id={dup_id} appears in customers table {len(customers[customers.customer_id == dup_id])} times")
print(f"Orders for this customer: {len(orders[orders.customer_id == dup_id])}")
print(f"Rows after merge:         {len(merged[merged.customer_id == dup_id])}  (orders x customer rows)")

In [ ]:
# DEFENSE: use validate= to catch duplicates before they explode
try:
    orders.merge(customers, on="customer_id", how="left", validate="m:1")
except pd.errors.MergeError as e:
    print(f"Caught it! MergeError: {e}")
    print("\nThis is exactly what we want — fail loud, fail early.")

## Pitfall 2: Left vs Inner — Silently Losing Rows

An `inner` join quietly drops every row whose key doesn't exist in the other table.
If you expected a left join but wrote `how='inner'` (the default!), you lose data
with no warning.

In [ ]:
# Remove duplicates first so we can focus on the join-type issue
customers_clean = customers.drop_duplicates(subset="customer_id", keep="first")

# How many orders reference a customer_id NOT in the customers table?
orphan_mask = ~orders["customer_id"].isin(customers_clean["customer_id"])
print(f"Orders with unknown customer_id: {orphan_mask.sum()} out of {len(orders)}")

# Inner join (pandas default!) silently drops them
inner = orders.merge(customers_clean, on="customer_id", how="inner")
left  = orders.merge(customers_clean, on="customer_id", how="left")

print(f"\nInner join rows: {len(inner)}  (lost {len(orders) - len(inner)} orders!)")
print(f"Left join rows:  {len(left)}   (kept all orders)")

In [ ]:
# DEFENSE: use indicator=True to audit what matched
audited = orders.merge(customers_clean, on="customer_id", how="left", indicator=True)

print("Merge audit (value_counts of _merge column):")
print(audited["_merge"].value_counts())
print("\n'left_only' = orders whose customer_id has no match in customers.")
display(audited[audited["_merge"] == "left_only"].head())

## Pitfall 3: Merging on Wrong Column Types

If one table has `customer_id` as `int64` and the other as `object` (string),
pandas will produce **zero matches** — silently.

In [ ]:
# Simulate a CSV import that read customer_id as string
customers_str = customers_clean.copy()
customers_str["customer_id"] = customers_str["customer_id"].astype(str)

print(f"orders.customer_id dtype:    {orders['customer_id'].dtype}")
print(f"customers.customer_id dtype: {customers_str['customer_id'].dtype}")

# Merge produces zero matches — no error, no warning
broken = orders.merge(customers_str, on="customer_id", how="inner")
print(f"\nRows after merge: {len(broken)}  <-- ZERO matches, completely silent!")

In [ ]:
# DEFENSE: check dtypes before merging
def check_merge_dtypes(left, right, on):
    """Raise if merge keys have incompatible dtypes."""
    lt = left[on].dtype
    rt = right[on].dtype
    if lt != rt:
        raise TypeError(
            f"Merge key '{on}' dtype mismatch: left={lt}, right={rt}. "
            f"Cast one side before merging."
        )
    print(f"dtypes match for '{on}': {lt}")

try:
    check_merge_dtypes(orders, customers_str, "customer_id")
except TypeError as e:
    print(f"Caught it! {e}")

# Fix: cast to matching type, then merge
customers_str["customer_id"] = customers_str["customer_id"].astype(int)
fixed = orders.merge(customers_str, on="customer_id", how="inner")
print(f"\nAfter dtype fix — rows: {len(fixed)}")

## The Defensive Merge Checklist

A reusable pattern that catches all three pitfalls before they corrupt your pipeline.

In [ ]:
def safe_merge(left, right, on, how="left", validate="m:1"):
    """Merge with built-in safety checks."""
    # 1. Dtype check
    lt, rt = left[on].dtype, right[on].dtype
    assert lt == rt, f"dtype mismatch on '{on}': {lt} vs {rt}"

    n_before = len(left)

    # 2. Merge with validation + indicator
    result = left.merge(right, on=on, how=how, validate=validate, indicator=True)

    # 3. Row count audit
    n_after = len(result)
    unmatched = (result["_merge"] == "left_only").sum()

    print(f"Rows: {n_before} -> {n_after}  |  Unmatched: {unmatched}")

    return result.drop(columns="_merge")

# Use the safe merge with clean data
result = safe_merge(orders, customers_clean, on="customer_id")
display(result.head())

## Takeaways

| Pitfall | Symptom | Defense |
| :--- | :--- | :--- |
| Duplicate keys in lookup table | Row count **increases** after merge | `validate="m:1"` |
| Wrong join type (inner vs left) | Row count **decreases**, data silently lost | `indicator=True` + audit `_merge` column |
| Mismatched dtypes on join key | Zero matches, empty result | Check `.dtype` on both sides before merge |

**Golden rule:** always compare `len()` before and after a merge. If the number changed
and you didn't expect it to, investigate before continuing.